# Producto Unidad 1 — AquaVigía
## Dimensión U1: Jhenderson Aaron Machaca Mamani

Pregunta (Brief S2): ¿Cual fue el pH y la turbidez promedio por estacion y por mes, y que valores se pueden esperar el proximo mes segun la tendencia historica?

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("aquavigia-u1-jhenderson")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/11 03:24:38 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/11 03:24:41 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


## 1. Arquitectura Big Data seleccionada

**Caso de negocio:** Las estaciones de monitoreo de cuencas, lagos y pozos generan
mediciones dispersas entre sensores automaticos (IoT) y muestreos de laboratorio,
sin un sistema que combine ambas fuentes para anticipar episodios de deterioro de
la calidad del agua antes de que se conviertan en un riesgo sanitario o ambiental.

**Clasificacion batch/streaming:** Ambos.
- Streaming: lecturas de sensores IoT (pH, temperatura, turbidez, oxigeno,
  conductividad) publicadas de forma continua por estacion, para alertas inmediatas.
- Batch: historico acumulado de mediciones (sensor_iot + laboratorio) para calcular
  tendencias mensuales y entrenar modelos predictivos (esta dimension U1).

**Arquitectura seleccionada:** Lambda — regla de decision (S1, 2.5.1): "si el caso
necesita historico + tiempo real -> Lambda". AquaVigia necesita explicitamente una
batch layer (tendencias, entrenamiento de modelos) y una speed layer (alertas en
vivo), combinadas en una serving layer comun (Grafana).

**Tecnologias propuestas:** Kafka (ingesta de sensores IoT) -> Spark Structured
Streaming (speed layer, alertas) + Spark Batch/Jupyter (batch layer, esta dimension)
-> Data Lake en Parquet (almacenamiento) -> Grafana (visualizacion).

**Supuestos y riesgos:** Supuesto: volumen creciente de mediciones, generadas de
forma continua por 180 estaciones. Riesgo: drift entre lo que reporta la speed
layer en vivo y lo que confirma el historico batch al dia siguiente.

## 2. Transformaciones distribuidas con PySpark (S2)

In [2]:
ORIGEN_DATOS = "./data"
ARTIFACTS = "./artifacts"

df_estaciones = spark.read.csv(f"{ORIGEN_DATOS}/estaciones_agua.csv", header=True, inferSchema=True)
df_mediciones = spark.read.parquet(f"{ORIGEN_DATOS}/mediciones_calidad_agua.parquet")

print("Estaciones:", df_estaciones.count())
print("Mediciones:", df_mediciones.count())
df_mediciones.printSchema()

Estaciones: 180
Mediciones: 1250000
root
 |-- medicion_id: long (nullable = true)
 |-- estacion_id: string (nullable = true)
 |-- fecha: string (nullable = true)
 |-- hora: string (nullable = true)
 |-- canal: string (nullable = true)
 |-- ph: double (nullable = true)
 |-- temperatura_c: double (nullable = true)
 |-- turbidez_ntu: double (nullable = true)
 |-- oxigeno_disuelto_mgl: double (nullable = true)
 |-- conductividad_us_cm: double (nullable = true)
 |-- solidos_disueltos_ppm: double (nullable = true)
 |-- nitratos_mgl: double (nullable = true)
 |-- fosfatos_mgl: double (nullable = true)
 |-- coliformes_fecales_cfu: double (nullable = true)
 |-- observaciones: string (nullable = true)



In [3]:
from pyspark.sql.functions import col

# Transformaciones encadenadas (select + filter) - NO ejecutan nada todavia
df_alerta = (
    df_mediciones
    .select("medicion_id", "estacion_id", "canal", "ph", "turbidez_ntu")
    .filter((col("canal") == "laboratorio") & (col("turbidez_ntu") > 50))
)
df_alerta  # solo muestra el tipo DataFrame[...], no ejecuta nada (evaluacion perezosa)

DataFrame[medicion_id: bigint, estacion_id: string, canal: string, ph: double, turbidez_ntu: double]

In [4]:
# Accion: aqui SI se ejecuta el plan completo
df_alerta.show(10)
print("Filas en alerta:", df_alerta.count())

+-----------+-----------+-----------+----+------------+
|medicion_id|estacion_id|      canal|  ph|turbidez_ntu|
+-----------+-----------+-----------+----+------------+
|      23736|   EST-0101|laboratorio|7.59|       55.48|
|      39041|   EST-0149|laboratorio|7.93|        50.9|
|     133890|   EST-0015|laboratorio|8.26|       62.31|
|     162238|   EST-0085|laboratorio|6.13|       51.41|
|     178127|   EST-0022|laboratorio|7.91|        57.5|
|     186864|   EST-0071|laboratorio|7.88|       50.85|
|     189978|   EST-0041|laboratorio|7.14|       51.95|
|     198371|   EST-0156|laboratorio|7.33|       58.48|
|     310611|   EST-0179|laboratorio|6.78|        60.1|
|     329646|   EST-0149|laboratorio|7.05|        65.7|
+-----------+-----------+-----------+----+------------+
only showing top 10 rows
Filas en alerta: 39


In [5]:
from pyspark.sql.functions import when, lit, current_date

df_mediciones = df_mediciones.withColumn(
    "calidad_ph",
    when((col("ph") >= 6.5) & (col("ph") <= 8.5), "Aceptable")
    .when((col("ph") >= 5.5) & (col("ph") < 6.5), "Acida")
    .otherwise("Fuera de rango")
).withColumn(
    "fuente_dataset", lit("Proyecto Sello - AquaVigia")
).withColumn(
    "fecha_procesado", current_date()
)

df_mediciones.select("estacion_id", "ph", "calidad_ph", "fuente_dataset", "fecha_procesado").show(5)

+-----------+----+----------+--------------------+---------------+
|estacion_id|  ph|calidad_ph|      fuente_dataset|fecha_procesado|
+-----------+----+----------+--------------------+---------------+
|   EST-0133| 7.8| Aceptable|Proyecto Sello - ...|     2026-09-11|
|   EST-0063|6.29|     Acida|Proyecto Sello - ...|     2026-09-11|
|   EST-0070|6.44|     Acida|Proyecto Sello - ...|     2026-09-11|
|   EST-0004|6.98| Aceptable|Proyecto Sello - ...|     2026-09-11|
|   EST-0127|7.51| Aceptable|Proyecto Sello - ...|     2026-09-11|
+-----------+----+----------+--------------------+---------------+
only showing top 5 rows


In [6]:
from pyspark.sql.functions import avg, count as spark_count

# Agrupacion y agregacion: promedio de ph/turbidez por estacion
df_resumen_estacion = df_mediciones.groupBy("estacion_id").agg(
    avg("ph").alias("ph_promedio"),
    avg("turbidez_ntu").alias("turbidez_promedio"),
    spark_count("*").alias("num_mediciones"),
)
df_resumen_estacion.orderBy("estacion_id").show(5)

+-----------+-----------------+-----------------+--------------+
|estacion_id|      ph_promedio|turbidez_promedio|num_mediciones|
+-----------+-----------------+-----------------+--------------+
|   EST-0001|7.194418100332518|6.066952436027167|          6917|
|   EST-0002|7.205068155452427|5.971837296983731|          6896|
|   EST-0003|7.201237322515204|5.997718052738311|          6902|
|   EST-0004|   7.191504134588|6.106753635585958|          7014|
|   EST-0005|7.209475887958408|6.073133121570884|          6926|
+-----------+-----------------+-----------------+--------------+
only showing top 5 rows


In [7]:
# Conteo de calidad_ph por canal
df_mediciones.groupBy("canal", "calidad_ph").agg(spark_count("*").alias("cantidad")).show()

+-----------+--------------+--------+
|      canal|    calidad_ph|cantidad|
+-----------+--------------+--------+
| sensor_iot|         Acida|  129466|
| sensor_iot|     Aceptable|  951726|
|laboratorio|     Aceptable|  129963|
|laboratorio|Fuera de rango|    2604|
| sensor_iot|Fuera de rango|   18656|
|laboratorio|         Acida|   17585|
+-----------+--------------+--------+



In [8]:
from pyspark.sql.window import Window

# Funcion ventana: promedio de turbidez por estacion, SIN colapsar filas
ventana_estacion = Window.partitionBy("estacion_id")
df_con_ventana = df_mediciones.withColumn(
    "turbidez_promedio_estacion", avg("turbidez_ntu").over(ventana_estacion)
)
df_con_ventana.filter(col("estacion_id") == "EST-0003").select(
    "estacion_id", "turbidez_ntu", "turbidez_promedio_estacion"
).show(5)

+-----------+------------+--------------------------+
|estacion_id|turbidez_ntu|turbidez_promedio_estacion|
+-----------+------------+--------------------------+
|   EST-0003|        0.19|         5.997718052738318|
|   EST-0003|        2.69|         5.997718052738318|
|   EST-0003|        8.51|         5.997718052738318|
|   EST-0003|        3.42|         5.997718052738318|
|   EST-0003|        8.24|         5.997718052738318|
+-----------+------------+--------------------------+
only showing top 5 rows


In [9]:
# Plan de ejecucion de df_alerta
df_alerta.explain(True)

== Parsed Logical Plan ==
'Filter 'and('`=`('canal, laboratorio), '`>`('turbidez_ntu, 50))
+- Project [medicion_id#25L, estacion_id#26, canal#29, ph#30, turbidez_ntu#32]
   +- Relation [medicion_id#25L,estacion_id#26,fecha#27,hora#28,canal#29,ph#30,temperatura_c#31,turbidez_ntu#32,oxigeno_disuelto_mgl#33,conductividad_us_cm#34,solidos_disueltos_ppm#35,nitratos_mgl#36,fosfatos_mgl#37,coliformes_fecales_cfu#38,observaciones#39] parquet

== Analyzed Logical Plan ==
medicion_id: bigint, estacion_id: string, canal: string, ph: double, turbidez_ntu: double
Filter ((canal#29 = laboratorio) AND (turbidez_ntu#32 > cast(50 as double)))
+- Project [medicion_id#25L, estacion_id#26, canal#29, ph#30, turbidez_ntu#32]
   +- Relation [medicion_id#25L,estacion_id#26,fecha#27,hora#28,canal#29,ph#30,temperatura_c#31,turbidez_ntu#32,oxigeno_disuelto_mgl#33,conductividad_us_cm#34,solidos_disueltos_ppm#35,nitratos_mgl#36,fosfatos_mgl#37,coliformes_fecales_cfu#38,observaciones#39] parquet

== Optimized Logic

In [10]:
import re
from operator import add

# RDD: conteo distribuido de palabras sobre observaciones (solo laboratorio)
rdd = df_mediciones.select("observaciones").rdd.map(lambda x: x.observaciones)
rdd = rdd.filter(lambda texto: texto is not None)

palabras = rdd.flatMap(
    lambda linea: re.sub(r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()).split()
)
pares = palabras.filter(lambda p: p != "").map(lambda palabra: (palabra, 1))
conteo = pares.reduceByKey(add)

conteo.takeOrdered(10, key=lambda x: -x[1])

[('sin', 117498),
 ('de', 63027),
 ('en', 40092),
 ('la', 29136),
 ('muestreo', 29084),
 ('olor', 29084),
 ('sedimentos', 28923),
 ('agua', 28814),
 ('algas', 28783),
 ('presencia', 28783)]

**Hallazgo (evaluacion perezosa):** `df_alerta` sola (celda de transformaciones) no
ejecuta nada -- Spark solo muestra `DataFrame[...]`. Recien con `.show()`/`.count()`
se dispara la ejecucion real, confirmando el comportamiento lazy explicado en S2 (2.5).

**Reflexion:** el `explain(True)` muestra que Catalyst movio el `Filter` antes del
`Project` (predicate pushdown) y agrego automaticamente `isnotnull(canal)` e
`isnotnull(turbidez_ntu)` -- optimizaciones que nunca escribimos explicitamente.

## 3. Calidad de datos y particionamiento analitico (S3)

In [11]:
from pyspark.sql.functions import col, count as spark_count, when

columnas_requeridas = {
    "medicion_id", "estacion_id", "fecha", "hora", "canal",
    "ph", "temperatura_c", "turbidez_ntu", "oxigeno_disuelto_mgl",
    "conductividad_us_cm", "solidos_disueltos_ppm",
}
faltantes = columnas_requeridas - set(df_mediciones.columns)
if faltantes:
    raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")
print("Esquema validado: columnas obligatorias presentes.")

Esquema validado: columnas obligatorias presentes.


In [12]:
# Nulos por columna
total_filas = df_mediciones.count()
nulos = df_mediciones.select([
    spark_count(when(col(c).isNull(), c)).alias(c) for c in
    ["ph","temperatura_c","turbidez_ntu","oxigeno_disuelto_mgl","conductividad_us_cm",
     "solidos_disueltos_ppm","nitratos_mgl","fosfatos_mgl","coliformes_fecales_cfu"]
]).collect()[0].asDict()
for columna, cantidad in nulos.items():
    print(f"{columna}: {cantidad} nulos ({cantidad/total_filas*100:.1f}%)")

ph: 0 nulos (0.0%)
temperatura_c: 0 nulos (0.0%)
turbidez_ntu: 0 nulos (0.0%)
oxigeno_disuelto_mgl: 0 nulos (0.0%)
conductividad_us_cm: 0 nulos (0.0%)
solidos_disueltos_ppm: 0 nulos (0.0%)
nitratos_mgl: 1099848 nulos (88.0%)
fosfatos_mgl: 1099848 nulos (88.0%)
coliformes_fecales_cfu: 1099848 nulos (88.0%)


**Filtrado — Tecnica 1 (expresion SQL como texto) y Tecnica 2 (expresion booleana con col()):**

In [13]:
# Filtrado - Tecnica 1: expresion SQL como texto
df_mediciones.filter("canal = 'laboratorio' AND turbidez_ntu > 50").show(5)

+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------+----------+--------------------+---------------+
|medicion_id|estacion_id|     fecha| hora|      canal|  ph|temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|       observaciones|calidad_ph|      fuente_dataset|fecha_procesado|
+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------+----------+--------------------+---------------+
|      23736|   EST-0101|2025-08-30|23:09|laboratorio|7.59|          9.6|       55.48|                8.04|              243.8|                168.2|        0.85|       0.463|                  53.

In [14]:
# Filtrado - Tecnica 2: expresion booleana con col(), incluye between() y eqNullSafe()
df_mediciones.filter((col("canal") == "laboratorio") & (col("turbidez_ntu").between(20, 60))).show(5)

# eqNullSafe: comparacion segura contra nulos
print("Mediciones con canal nulo (eqNullSafe):", df_mediciones.filter(col("canal").eqNullSafe(None)).count())

+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------+----------+--------------------+---------------+
|medicion_id|estacion_id|     fecha| hora|      canal|  ph|temperatura_c|turbidez_ntu|oxigeno_disuelto_mgl|conductividad_us_cm|solidos_disueltos_ppm|nitratos_mgl|fosfatos_mgl|coliformes_fecales_cfu|       observaciones|calidad_ph|      fuente_dataset|fecha_procesado|
+-----------+-----------+----------+-----+-----------+----+-------------+------------+--------------------+-------------------+---------------------+------------+------------+----------------------+--------------------+----------+--------------------+---------------+
|        275|   EST-0151|2024-04-22|09:11|laboratorio|7.61|          4.3|        22.7|                7.18|              557.4|                355.3|        6.73|       0.722|                   7.

**Orden — Tecnica 1 (orderBy()) y Tecnica 2 (sort()):**

In [15]:
# Orden - Tecnica 1: orderBy(), por dos columnas
df_mediciones.orderBy(col("estacion_id").asc(), col("turbidez_ntu").desc()).select(
    "estacion_id", "turbidez_ntu", "ph"
).show(5)

+-----------+------------+----+
|estacion_id|turbidez_ntu|  ph|
+-----------+------------+----+
|   EST-0001|       51.13|7.64|
|   EST-0001|       49.24|7.55|
|   EST-0001|       46.31|7.24|
|   EST-0001|       45.76|7.31|
|   EST-0001|       44.46|7.48|
+-----------+------------+----+
only showing top 5 rows


In [16]:
# Orden - Tecnica 2: sort() (alias de orderBy)
df_mediciones.sort(col("ph").desc_nulls_last()).select("estacion_id", "ph").show(5)

+-----------+---+
|estacion_id| ph|
+-----------+---+
|   EST-0018|9.5|
|   EST-0156|9.5|
|   EST-0176|9.5|
|   EST-0117|9.5|
|   EST-0075|9.5|
+-----------+---+
only showing top 5 rows


**Duplicados — diagnostico y tratamiento con 2 tecnicas (incluida Window+row_number):**

In [17]:
dup = df_mediciones.groupBy("estacion_id", "fecha", "hora").count().filter("count > 1")
print("Grupos duplicados (estacion_id+fecha+hora):", dup.count())

[Stage 42:================================================>         (5 + 1) / 6]

Grupos duplicados (estacion_id+fecha+hora): 4175


In [18]:
total = df_mediciones.count()
sin_dup_fila_completa = df_mediciones.distinct().count()
print(f"Total: {total}, sin duplicar (fila completa): {sin_dup_fila_completa}")

[Stage 53:==================================================>       (7 + 1) / 8]

Total: 1250000, sin duplicar (fila completa): 1250000


In [19]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

ventana_dedup = Window.partitionBy("estacion_id", "fecha", "hora").orderBy(col("medicion_id").desc())
df_mediciones_unico = (
    df_mediciones
    .withColumn("row_num", row_number().over(ventana_dedup))
    .filter(col("row_num") == 1)
    .drop("row_num")
)
print(f"Mediciones: {df_mediciones.count():,} -> tras deduplicar: {df_mediciones_unico.count():,}")

[Stage 60:================================================>         (5 + 1) / 6]

Mediciones: 1,250,000 -> tras deduplicar: 1,245,817


**Nulos: tratamiento con criterio documentado por columna:**

In [20]:
# nitratos_mgl, fosfatos_mgl, coliformes_fecales_cfu son NULL de forma legitima
# (solo laboratorio los mide) -- no se rellenan con na.fill(), pues inventar un valor
# seria peor que dejarlos nulos. Solo se eliminan filas sin identificador critico:
df_valido = df_mediciones_unico.na.drop(subset=["medicion_id", "estacion_id", "fecha", "hora"])
print(f"Filas antes: {df_mediciones_unico.count():,}, despues de na.drop(subset criticos): {df_valido.count():,}")

[Stage 72:================================================>         (5 + 1) / 6]

Filas antes: 1,245,817, despues de na.drop(subset criticos): 1,245,817


In [21]:
# Integrar con estaciones (Silver) y particionar por 'region' (baja cardinalidad, usada en filtros)
df_estaciones_limpio = df_estaciones.dropDuplicates(["estacion_id"])
df_gold_base = df_valido.join(df_estaciones_limpio, on="estacion_id", how="left")
print("Columnas Gold:", df_gold_base.columns)
print("Filas:", df_gold_base.count())
df_gold_base = df_gold_base.cache()

Columnas Gold: ['estacion_id', 'medicion_id', 'fecha', 'hora', 'canal', 'ph', 'temperatura_c', 'turbidez_ntu', 'oxigeno_disuelto_mgl', 'conductividad_us_cm', 'solidos_disueltos_ppm', 'nitratos_mgl', 'fosfatos_mgl', 'coliformes_fecales_cfu', 'observaciones', 'calidad_ph', 'fuente_dataset', 'fecha_procesado', 'nombre_estacion', 'region', 'cuenca', 'tipo_fuente', 'latitud', 'longitud', 'fecha_instalacion']


Filas: 1245817


In [22]:
ruta_gold = f"{ARTIFACTS}/mediciones_particionado"

(
    df_gold_base
    .repartition(4)
    .write.format("parquet")
    .mode("overwrite")
    .partitionBy("region")
    .save(ruta_gold)
)

import os
print("Particiones (region):", sorted(os.listdir(ruta_gold)))

Particiones (region): ['._SUCCESS.crc', '_SUCCESS', 'region=Arequipa', 'region=Cusco', 'region=Junín', 'region=Lambayeque', 'region=Lima', 'region=Loreto', 'region=Piura', 'region=Puno']


In [23]:
df_verificacion = spark.read.parquet(ruta_gold)
assert df_verificacion.count() == df_gold_base.count()
print(f"Verificado: {df_verificacion.count():,} filas, ida y vuelta sin perdida.")

Verificado: 1,245,817 filas, ida y vuelta sin perdida.


In [24]:
# PartitionFilters: filtrar por la columna de particion
df_verificacion.filter(col("region") == "Puno").explain(True)

== Parsed Logical Plan ==
'Filter '`=`('region, Puno)
+- Relation [estacion_id#1509,medicion_id#1510L,fecha#1511,hora#1512,canal#1513,ph#1514,temperatura_c#1515,turbidez_ntu#1516,oxigeno_disuelto_mgl#1517,conductividad_us_cm#1518,solidos_disueltos_ppm#1519,nitratos_mgl#1520,fosfatos_mgl#1521,coliformes_fecales_cfu#1522,observaciones#1523,calidad_ph#1524,fuente_dataset#1525,fecha_procesado#1526,nombre_estacion#1527,cuenca#1528,tipo_fuente#1529,latitud#1530,longitud#1531,fecha_instalacion#1532,region#1533] parquet

== Analyzed Logical Plan ==
estacion_id: string, medicion_id: bigint, fecha: string, hora: string, canal: string, ph: double, temperatura_c: double, turbidez_ntu: double, oxigeno_disuelto_mgl: double, conductividad_us_cm: double, solidos_disueltos_ppm: double, nitratos_mgl: double, fosfatos_mgl: double, coliformes_fecales_cfu: double, observaciones: string, calidad_ph: string, fuente_dataset: string, fecha_procesado: date, nombre_estacion: string, cuenca: string, tipo_fuente: 

**Capas Bronze/Silver/Gold en este pipeline:**
- **Bronze:** `estaciones_agua.csv` / `mediciones_calidad_agua.parquet` tal como llegan.
- **Silver:** `df_valido` — esquema validado, duplicados resueltos (Window+row_number),
  nulos criticos tratados.
- **Gold:** `mediciones_particionado/` — integrado con estaciones y particionado por
  `region`, listo para el componente ML (Bloque 4).

## 4. Componente ML distribuido (S4)

**Objetivo (Brief):** proyectar `ph_promedio` y `turbidez_promedio` del mes siguiente
por estacion, a partir del historico agregado mensual (Gold del Bloque 3).

In [25]:
from pyspark.sql.functions import to_date, date_format, dense_rank, lead

df_mes = df_verificacion.withColumn("AnioMes", date_format(to_date(col("fecha")), "yyyy-MM"))

df_agregado = (
    df_mes.groupBy("estacion_id", "AnioMes")
    .agg(
        avg("ph").alias("ph_promedio"),
        avg("turbidez_ntu").alias("turbidez_promedio"),
        avg("temperatura_c").alias("temperatura_promedio"),
        avg("oxigeno_disuelto_mgl").alias("oxigeno_promedio"),
        avg("conductividad_us_cm").alias("conductividad_promedio"),
        avg("solidos_disueltos_ppm").alias("solidos_promedio"),
        spark_count("*").alias("cantidad_mediciones"),
    )
)
print("Filas agregadas (estacion x mes):", df_agregado.count())
df_agregado.orderBy("estacion_id", "AnioMes").show(5)

Filas agregadas (estacion x mes): 4320


[Stage 113:================================================>      (14 + 2) / 16]

+-----------+-------+-----------------+------------------+--------------------+-----------------+----------------------+------------------+-------------------+
|estacion_id|AnioMes|      ph_promedio| turbidez_promedio|temperatura_promedio| oxigeno_promedio|conductividad_promedio|  solidos_promedio|cantidad_mediciones|
+-----------+-------+-----------------+------------------+--------------------+-----------------+----------------------+------------------+-------------------+
|   EST-0001|2024-01|7.213762376237623| 6.070693069306929|  15.969966996699668|6.815379537953795|    388.61914191419146|248.18943894389437|                303|
|   EST-0001|2024-02|7.169689922480621|  6.20658914728682|  16.233720930232558|7.023875968992248|    374.56317829457373|241.01666666666674|                258|
|   EST-0001|2024-03|7.210361010830325| 5.943610108303249|    15.5129963898917|7.002563176895305|    377.40324909747284|241.67689530685925|                277|
|   EST-0001|2024-04|7.190899653979238|6

In [26]:
ventana_mes = Window.orderBy("AnioMes")
mapa_meses = df_agregado.select("AnioMes").distinct().withColumn("mes_numero", dense_rank().over(ventana_mes))
df_agregado = df_agregado.join(mapa_meses, on="AnioMes", how="left")

ventana_lead = Window.partitionBy("estacion_id").orderBy("mes_numero")
df_con_objetivo = (
    df_agregado
    .withColumn("ph_promedio_siguiente", lead("ph_promedio", 1).over(ventana_lead))
    .withColumn("turbidez_promedio_siguiente", lead("turbidez_promedio", 1).over(ventana_lead))
)

df_dataset = df_con_objetivo.na.drop(subset=["ph_promedio_siguiente", "turbidez_promedio_siguiente"])
print("Filas utilizables (con mes siguiente conocido):", df_dataset.count())
df_dataset = df_dataset.cache()
df_dataset.orderBy("estacion_id", "mes_numero").select(
    "estacion_id", "mes_numero", "ph_promedio", "ph_promedio_siguiente"
).show(10)

26/09/11 03:28:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:25 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
          

Filas utilizables (con mes siguiente conocido): 4140


26/09/11 03:28:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/09/11 03:28:27 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
          

+-----------+----------+-----------------+---------------------+
|estacion_id|mes_numero|      ph_promedio|ph_promedio_siguiente|
+-----------+----------+-----------------+---------------------+
|   EST-0001|         1|7.213762376237623|    7.169689922480621|
|   EST-0001|         2|7.169689922480621|    7.210361010830325|
|   EST-0001|         3|7.210361010830325|    7.190899653979238|
|   EST-0001|         4|7.190899653979238|    7.216031746031747|
|   EST-0001|         5|7.216031746031747|     7.22836956521739|
|   EST-0001|         6| 7.22836956521739|    7.204771241830064|
|   EST-0001|         7|7.204771241830064|    7.217482517482519|
|   EST-0001|         8|7.217482517482519|    7.226449275362318|
|   EST-0001|         9|7.226449275362318|    7.220034602076125|
|   EST-0001|        10|7.220034602076125|    7.131942446043165|
+-----------+----------+-----------------+---------------------+
only showing top 10 rows


In [28]:
PREDICTORES = ["mes_numero", "ph_promedio", "turbidez_promedio", "temperatura_promedio",
               "oxigeno_promedio", "conductividad_promedio", "solidos_promedio"]
OBJETIVO = "ph_promedio_siguiente"

for c in PREDICTORES:
    print(f"{c:24s} correlacion con {OBJETIVO}: {df_dataset.stat.corr(c, OBJETIVO):.4f}")

mes_numero               correlacion con ph_promedio_siguiente: 0.0093
ph_promedio              correlacion con ph_promedio_siguiente: 0.0130
turbidez_promedio        correlacion con ph_promedio_siguiente: 0.0203
temperatura_promedio     correlacion con ph_promedio_siguiente: 0.0298
oxigeno_promedio         correlacion con ph_promedio_siguiente: -0.0327
conductividad_promedio   correlacion con ph_promedio_siguiente: 0.0016
solidos_promedio         correlacion con ph_promedio_siguiente: -0.0002


Como el objetivo es un valor FUTURO (mes siguiente), el split debe ser cronologico,
no aleatorio (randomSplit) — igual que advierte la guia S4 para el caso de series de
tiempo: un split aleatorio filtraria informacion del futuro hacia el entrenamiento.

In [29]:
max_mes = df_dataset.agg({"mes_numero": "max"}).collect()[0][0]
corte = max_mes - 4
print("mes_numero maximo:", max_mes, " corte de train/test:", corte)

df_train = df_dataset.filter(col("mes_numero") <= corte)
df_test = df_dataset.filter(col("mes_numero") > corte)
print("Train:", df_train.count(), "Test:", df_test.count())

mes_numero maximo: 23  corte de train/test: 19
Train: 3420 Test: 720


In [31]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

ensamblador = VectorAssembler(inputCols=PREDICTORES, outputCol="features")
df_train_ml = ensamblador.transform(df_train).select("features", OBJETIVO)
df_test_ml = ensamblador.transform(df_test).select("features", OBJETIVO)

def evaluar(pred, nombre):
    res = {}
    for m in ["rmse", "r2", "mae"]:
        ev = RegressionEvaluator(labelCol=OBJETIVO, predictionCol="prediction", metricName=m)
        res[m.upper()] = ev.evaluate(pred)
    print(f"{nombre}: RMSE={res['RMSE']:.4f}  R2={res['R2']:.4f}  MAE={res['MAE']:.4f}")
    return res

lr_base = LinearRegression(featuresCol="features", labelCol=OBJETIVO)
modelo_base = lr_base.fit(df_train_ml)
print("Coeficientes:", modelo_base.coefficients)
print("Intercepto:", modelo_base.intercept)
resultados_base = evaluar(modelo_base.transform(df_test_ml), "LinearRegression base")

26/09/11 03:36:57 WARN Instrumentation: [d08735ac] regParam is zero, which might cause numerical instability and overfitting.


Coeficientes: [-1.9858874270433598e-05,0.010521332345309475,0.0015531946837532252,0.001086920564762031,-0.013169098455433376,0.00016526649841551872,-0.00020848724693148856]
Intercepto: 7.177742152811812
LinearRegression base: RMSE=0.0355  R2=0.0008  MAE=0.0283


In [33]:
configuraciones = [
    {"nombre": "Sin regularizacion", "regParam": 0.0, "elasticNetParam": 0.0},
    {"nombre": "Ridge (L2)", "regParam": 0.1, "elasticNetParam": 0.0},
    {"nombre": "Elastic Net (L1+L2)", "regParam": 0.1, "elasticNetParam": 0.5},
]
comparacion_configs = []
for config in configuraciones:
    lr = LinearRegression(featuresCol="features", labelCol=OBJETIVO,
                           regParam=config["regParam"], elasticNetParam=config["elasticNetParam"])
    modelo = lr.fit(df_train_ml)
    r = evaluar(modelo.transform(df_test_ml), config["nombre"])
    r["Configuracion"] = config["nombre"]
    comparacion_configs.append(r)

import pandas as pd
pd.DataFrame(comparacion_configs)[["Configuracion","RMSE","R2","MAE"]]

26/09/11 03:37:21 WARN Instrumentation: [d323f501] regParam is zero, which might cause numerical instability and overfitting.


Sin regularizacion: RMSE=0.0355  R2=0.0008  MAE=0.0283
Ridge (L2): RMSE=0.0355  R2=-0.0010  MAE=0.0284
Elastic Net (L1+L2): RMSE=0.0355  R2=-0.0021  MAE=0.0284


,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.035451,0.000849,0.028320
1,Ridge (L2),0.035483,-0.000979,0.028351
2,Elastic Net (L1+L2),0.035502,-0.002050,0.028370


In [34]:
from pyspark.ml.regression import RandomForestRegressor

rf = RandomForestRegressor(featuresCol="features", labelCol=OBJETIVO, numTrees=50, maxDepth=6, seed=42)
modelo_rf = rf.fit(df_train_ml)
resultados_rf = evaluar(modelo_rf.transform(df_test_ml), "Random Forest")

importancias = sorted(zip(PREDICTORES, modelo_rf.featureImportances.toArray()), key=lambda x: -x[1])
for v, imp in importancias:
    print(f"{v:24s} {imp:.4f}")

Random Forest: RMSE=0.0356  R2=-0.0063  MAE=0.0284
oxigeno_promedio         0.1654
turbidez_promedio        0.1626
ph_promedio              0.1479
temperatura_promedio     0.1438
mes_numero               0.1383
conductividad_promedio   0.1248
solidos_promedio         0.1173


In [36]:
tabla_final = pd.DataFrame(comparacion_configs + [{"Configuracion":"Random Forest", **resultados_rf}])[["Configuracion","RMSE","R2","MAE"]]
tabla_final

,Configuracion,RMSE,R2,MAE
0,Sin regularizacion,0.035451,0.000849,0.028320
1,Ridge (L2),0.035483,-0.000979,0.028351
2,Elastic Net (L1+L2),0.035502,-0.002050,0.028370
3,Random Forest,0.035577,-0.006250,0.028437


**Evaluation (conclusion de negocio):** el modelo lineal sin regularizacion tiene el
mejor RMSE/MAE de los cuatro, aunque su R2 es practicamente cero. Esto no significa que
el modelo sea inutil: el pH promedio mensual por estacion es un valor muy estable
(oscila apenas entre 7.1 y 7.3), asi que el error absoluto ya es pequeno (RMSE≈0.035,
menos del 2% del rango aceptable 6.5-8.5) incluso sin que el modelo capture una
tendencia real mas alla del promedio historico. **Validacion de negocio:** el modelo
SI sirve como referencia de "valor esperado" para el tecnico (decision del Brief),
pero no debe interpretarse como una prediccion de una tendencia real de deterioro —
para eso, el historico disponible (24 meses) es corto y la variabilidad mes a mes ya
es baja de por si.

In [37]:
modelo_ganador = modelo_base  # Sin regularizacion: mejor RMSE y unico con R2 positivo
modelo_ganador.write().overwrite().save(f"{ARTIFACTS}/modelo_ph_siguiente_mes")
print("Modelo guardado en", f"{ARTIFACTS}/modelo_ph_siguiente_mes")

Modelo guardado en ./artifacts/modelo_ph_siguiente_mes


## Error o hallazgo

**Que ocurrio:** al ensamblar el pipeline completo (S1 a S4) en un solo notebook, la
primera corrida del Bloque 4 fallo porque `ORIGEN_DATOS`/`ARTIFACTS` se habian
definido dentro del Bloque 2 y el kernel se reinicio antes de llegar al Bloque 4.

**Como lo identifique:** el traceback senalo un `NameError` en la celda de
`VectorAssembler`, la misma familia de error que ya habiamos visto en el notebook de S04.

**Como lo resolvi:** ejecute el notebook completo de punta a punta con
Kernel -> Restart & Run All, en vez de correr celdas sueltas fuera de orden — la misma
buena practica que ya habiamos identificado antes, ahora aplicada a un pipeline de
4 bloques en vez de uno solo.

## Reflexion tecnica breve

Integrar S1-S4 en un solo notebook reproducible confirma por que la guia S3 insiste en
capas Bronze/Silver/Gold: el Bloque 4 no vuelve a limpiar nada, solo agrega sobre la
salida Gold ya validada del Bloque 3 — si el dato de entrada no fuera confiable, el
modelo entrenaria sobre ruido sin saberlo. La comparacion sistematica de 4
configuraciones (Bloque 4) es lo que permite concluir, con evidencia y no por
intuicion, que el pH promedio mensual es demasiado estable para que un modelo de
regresion simple aporte mas que el propio promedio historico — un hallazgo honesto,
no un fracaso del pipeline.